# Diagnóstico de calidad de datos: estrés canino en guarderías

**Actividad:** GA1-220501094-01-AA1-EV05 – Diagnosticar la calidad de datos: preparación, exploración y documentación
**Rol:** analista de datos (desarrollo individual)
**Aprendiz:** Laura Vanessa Henao López

> **Nota sobre los datos:** los archivos de `data/raw/` son datos de **ejemplo simulados** (generados con
> `scripts/generar_datos_ejemplo.py`), con problemas de calidad incluidos a propósito. Para el proyecto real,
> se reemplazan por las observaciones recolectadas en la guardería y el notebook se ejecuta igual.
> Las cifras de las interpretaciones se calculan con código a partir de los DataFrames.

---
## Momento 1. Reconocimiento del proyecto y de los datasets

### 1.1 Proyecto y pregunta analítica
**Proyecto:** Bienestar canino en guarderías y hoteles para perros.

**Pregunta analítica:** ¿Existe relación entre el número de perros por espacio y los días de estadía con el nivel de estrés observado en perros alojados en una guardería canina?

### 1.2 Datasets seleccionados

| N.º | Archivo | DataFrame | Información que contiene | Relación con la pregunta |
|---|---|---|---|---|
| 1 | `perros_ingreso.csv` | `perros` | Una fila por perro: edad, sexo, si es su primera vez, fecha de ingreso y días de estadía | Datos de control y duración de la estadía |
| 2 | `observaciones_estres.csv` | `obs` | Una fila por perro y día: perros en el espacio, señales de estrés y nivel de estrés (0 a 3) | Contiene los factores (perros por espacio, día de estadía) y el resultado (nivel de estrés) |

### 1.3 Aspectos de calidad a revisar
Valores faltantes, tipos de datos coherentes, duplicados, valores fuera de rango (por ejemplo, un nivel de estrés fuera de la escala 0 a 3),
categorías escritas de forma distinta y coherencia entre ambos archivos (todo `perro_id` observado debe tener ficha de ingreso).

---
## Momento 2. Preparación del entorno y carga de datos

In [4]:
%pip install pandas numpy
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)

def md(texto):
    display(Markdown(texto))

RAW = Path("..") / "data" / "raw"
perros = pd.read_csv(RAW / "perros_ingreso.csv")
obs = pd.read_csv(RAW / "observaciones_estres.csv")

datasets = {"perros": perros, "obs": obs}
for nombre, df in datasets.items():
    print(f"{nombre}: cargado con {df.shape[0]} filas y {df.shape[1]} columnas")

Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-3.1.1-cp314-cp314-win_amd64.whl.metadata (16 kB)
  Using cached numpy-2.5.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
Using cached pandas-3.1.1-cp314-cp314-win_amd64.whl (10.9 MB)
Using cached numpy-2.5.4-cp314-cp314-win_amd64.whl (12.7 MB)

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   -------------------------------------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


perros: cargado con 61 filas y 6 columnas
obs: cargado con 362 filas y 9 columnas


---
## Momento 3. Inspección de la estructura
### 3.1 Dataset 1: `perros`

In [5]:
perros.head()

,perro_id,edad_anios,sexo,primera_vez_guarderia,fecha_ingreso,dias_estadia_total
0,P001,2.0,Hembra,No,06/09/2026,5
1,P002,10.0,Hembra,No,24/09/2026,9
2,P003,8.0,Macho,No,04/09/2026,2
3,P004,6.0,Hembra,No,26/09/2026,10
4,P005,6.0,Macho,No,26/09/2026,6


In [6]:
perros.tail()

,perro_id,edad_anios,sexo,primera_vez_guarderia,fecha_ingreso,dias_estadia_total
56,P057,7.0,Macho,No,04/10/2026,3
57,P058,2.0,Macho,No,18/09/2026,10
58,P059,9.0,Macho,No,12/09/2026,2
59,P060,9.0,Macho,No,01/09/2026,8
60,P005,6.0,Macho,No,26/09/2026,6


In [7]:
print("shape:", perros.shape)
print("columns:", list(perros.columns))
perros.dtypes

shape: (61, 6)
columns: ['perro_id', 'edad_anios', 'sexo', 'primera_vez_guarderia', 'fecha_ingreso', 'dias_estadia_total']


perro_id                     str
edad_anios               float64
sexo                         str
primera_vez_guarderia        str
fecha_ingreso                str
dias_estadia_total         int64
dtype: object

In [8]:
perros.info()

<class 'pandas.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   perro_id               61 non-null     str    
 1   edad_anios             56 non-null     float64
 2   sexo                   61 non-null     str    
 3   primera_vez_guarderia  57 non-null     str    
 4   fecha_ingreso          61 non-null     str    
 5   dias_estadia_total     61 non-null     int64  
dtypes: float64(1), int64(1), str(4)
memory usage: 3.0 KB


In [9]:
perros.describe()

,edad_anios,dias_estadia_total
count,56.000000,61.000000
mean,7.285714,6.000000
std,6.210590,2.750757
min,-3.000000,2.000000
25%,3.750000,4.000000
50%,7.000000,6.000000
75%,10.000000,8.000000
max,45.000000,10.000000


In [10]:
perros.describe(include='object')

C:\Users\Vanessa Henao\AppData\Local\Temp\ipykernel_2600\1244522916.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  perros.describe(include='object')


,perro_id,sexo,primera_vez_guarderia,fecha_ingreso
count,61,61,57,61
unique,60,4,2,30
top,P005,Hembra,No,24/09/2026
freq,2,30,35,4


### 3.2 Dataset 2: `obs`

In [11]:
obs.head()

,obs_id,perro_id,dia_estadia,perros_en_espacio,jadeo,temblor,ladrido_excesivo,rechazo_alimento,nivel_estres
0,1,P001,1,16,0.0,0,1,0,1.0
1,2,P001,2,16,1.0,0,0,0,1.0
2,3,P001,3,NaN,NaN,0,1,1,1.0
3,4,P001,4,8,1.0,0,1,0,1.0
4,5,P001,5,19,1.0,0,0,0,1.0


In [12]:
obs.tail()

,obs_id,perro_id,dia_estadia,perros_en_espacio,jadeo,temblor,ladrido_excesivo,rechazo_alimento,nivel_estres
357,358,P060,6,16,1.0,0,0,0,1.0
358,359,P060,7,18,0.0,0,0,0,1.0
359,360,P060,8,10,0.0,0,0,0,0.0
360,11,P002,6,9,0.0,0,0,0,0.0
361,56,P011,5,7,0.0,0,0,0,1.0


In [13]:
print("shape:", obs.shape)
print("columns:", list(obs.columns))
obs.dtypes

shape: (362, 9)
columns: ['obs_id', 'perro_id', 'dia_estadia', 'perros_en_espacio', 'jadeo', 'temblor', 'ladrido_excesivo', 'rechazo_alimento', 'nivel_estres']


obs_id                 int64
perro_id                 str
dia_estadia            int64
perros_en_espacio        str
jadeo                float64
temblor                int64
ladrido_excesivo       int64
rechazo_alimento       int64
nivel_estres         float64
dtype: object

In [14]:
obs.info()

<class 'pandas.DataFrame'>
RangeIndex: 362 entries, 0 to 361
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   obs_id             362 non-null    int64  
 1   perro_id           362 non-null    str    
 2   dia_estadia        362 non-null    int64  
 3   perros_en_espacio  350 non-null    str    
 4   jadeo              359 non-null    float64
 5   temblor            362 non-null    int64  
 6   ladrido_excesivo   362 non-null    int64  
 7   rechazo_alimento   362 non-null    int64  
 8   nivel_estres       356 non-null    float64
dtypes: float64(2), int64(5), str(2)
memory usage: 25.6 KB


In [15]:
obs.describe()

,obs_id,dia_estadia,jadeo,temblor,ladrido_excesivo,rechazo_alimento,nivel_estres
count,362.000000,362.000000,359.000000,362.000000,362.000000,362.000000,356.000000
mean,179.687845,4.110497,0.259053,0.146409,0.243094,0.168508,0.915730
std,104.364078,2.540301,0.438726,0.354005,0.429545,0.374835,0.771813
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000
25%,89.250000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,179.500000,4.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,269.750000,6.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,360.000000,10.000000,1.000000,1.000000,1.000000,1.000000,5.000000


In [16]:
obs.describe(include='object')

C:\Users\Vanessa Henao\AppData\Local\Temp\ipykernel_2600\785054221.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obs.describe(include='object')


,perro_id,perros_en_espacio
count,362,350
unique,61,21
top,P002,15
freq,10,26


### 3.3 Tabla de diagnóstico estructural

In [17]:
estructura = pd.DataFrame({
    "Dataset": list(datasets),
    "Filas": [d.shape[0] for d in datasets.values()],
    "Columnas": [d.shape[1] for d in datasets.values()],
    "Tipos de datos": [", ".join(f"{t}: {n}" for t, n in d.dtypes.astype(str).value_counts().items()) for d in datasets.values()],
    "Filas duplicadas": [int(d.duplicated().sum()) for d in datasets.values()],
})
estructura

,Dataset,Filas,Columnas,Tipos de datos,Filas duplicadas
0,perros,61,6,"str: 4, float64: 1, int64: 1",1
1,obs,362,9,"int64: 5, str: 2, float64: 2",2


---
## Momento 4. Diagnóstico inicial de calidad
### 4.1 Valores faltantes

In [18]:
for nombre, df in datasets.items():
    nulos = df.isna().sum()
    tabla = pd.DataFrame({"nulos": nulos, "% del total": (nulos / len(df) * 100).round(1)})
    md(f"**`{nombre}`** — columnas con valores faltantes:")
    display(tabla[tabla["nulos"] > 0])

**`perros`** — columnas con valores faltantes:

,nulos,% del total
edad_anios,5,8.2
primera_vez_guarderia,4,6.6


**`obs`** — columnas con valores faltantes:

,nulos,% del total
perros_en_espacio,12,3.3
jadeo,3,0.8
nivel_estres,6,1.7


### 4.2 Tipos de datos

In [19]:
numericas_esperadas = ["perros_en_espacio", "dia_estadia", "nivel_estres"]
for col in numericas_esperadas:
    print(f"{col}: tipo actual = {obs[col].dtype}")

no_numericos = pd.to_numeric(obs["perros_en_espacio"], errors="coerce").isna() & obs["perros_en_espacio"].notna()
md(f"En `perros_en_espacio` hay **{no_numericos.sum()}** valores con texto en lugar de número:")
display(obs.loc[no_numericos, ["obs_id", "perro_id", "perros_en_espacio"]])
md(f"`fecha_ingreso` en `perros` es de tipo **{perros['fecha_ingreso'].dtype}** (texto); debería ser fecha.")

perros_en_espacio: tipo actual = str
dia_estadia: tipo actual = int64
nivel_estres: tipo actual = float64


En `perros_en_espacio` hay **2** valores con texto en lugar de número:

,obs_id,perro_id,perros_en_espacio
273,274,P046,12 perros
282,283,P047,ocho


`fecha_ingreso` en `perros` es de tipo **str** (texto); debería ser fecha.

### 4.3 Posibles inconsistencias

In [20]:
obs_num = obs.assign(perros_en_espacio=pd.to_numeric(obs["perros_en_espacio"], errors="coerce"))

fuera_escala = obs[(obs["nivel_estres"] < 0) | (obs["nivel_estres"] > 3)]
dia_invalido = obs[obs["dia_estadia"] < 1]
atipicos = obs_num[obs_num["perros_en_espacio"] > 30]
edad_invalida = perros[(perros["edad_anios"] < 0) | (perros["edad_anios"] > 25)]
sin_ficha = obs[~obs["perro_id"].isin(perros["perro_id"])]

md(f"- `nivel_estres` fuera de la escala 0 a 3: **{len(fuera_escala)}** registros")
display(fuera_escala[["obs_id", "perro_id", "nivel_estres"]])
md(f"- `dia_estadia` menor que 1: **{len(dia_invalido)}** registro(s)")
display(dia_invalido[["obs_id", "perro_id", "dia_estadia"]])
md(f"- `perros_en_espacio` mayor que 30 (atípico): **{len(atipicos)}** registro(s)")
display(atipicos[["obs_id", "perros_en_espacio"]])
md(f"- `edad_anios` negativa o mayor a 25: **{len(edad_invalida)}** registro(s)")
display(edad_invalida[["perro_id", "edad_anios"]])
md(f"- Observaciones con `perro_id` sin ficha de ingreso: **{len(sin_ficha)}**")
display(sin_ficha[["obs_id", "perro_id"]])
md("- Categorías de `sexo` escritas de forma distinta:")
display(perros["sexo"].value_counts())

- `nivel_estres` fuera de la escala 0 a 3: **2** registros

,obs_id,perro_id,nivel_estres
191,192,P034,-1.0
295,296,P049,5.0


- `dia_estadia` menor que 1: **1** registro(s)

,obs_id,perro_id,dia_estadia
140,141,P024,0


- `perros_en_espacio` mayor que 30 (atípico): **1** registro(s)

,obs_id,perros_en_espacio
165,166,120.0


... (2 outputs left out)

- Observaciones con `perro_id` sin ficha de ingreso: **1**

,obs_id,perro_id
329,330,P999


- Categorías de `sexo` escritas de forma distinta:

sexo
Hembra    30
Macho     29
macho      1
HEMBRA     1
Name: count, dtype: int64

### 4.4 Limitaciones frente a la pregunta analítica

In [21]:
validas = obs_num.dropna(subset=["perros_en_espacio", "nivel_estres"])
validas = validas[validas["nivel_estres"].between(0, 3) & (validas["dia_estadia"] >= 1) & (validas["perros_en_espacio"] <= 30)]
md(f"De **{len(obs)}** observaciones, **{len(validas)}** ({len(validas)/len(obs)*100:.1f} %) tienen las dos variables clave "
   f"(`perros_en_espacio` y `nivel_estres`) completas y dentro de rango.")
md(f"Perros con ficha: **{perros['perro_id'].nunique()}**. Por tratarse de una sola guardería, los resultados no se pueden generalizar a otras.")

De **362** observaciones, **338** (93.4 %) tienen las dos variables clave (`perros_en_espacio` y `nivel_estres`) completas y dentro de rango.

Perros con ficha: **60**. Por tratarse de una sola guardería, los resultados no se pueden generalizar a otras.

In [22]:
md("Distribución de `nivel_estres` (solo valores dentro de la escala):")
display(obs["nivel_estres"][obs["nivel_estres"].between(0, 3)].value_counts().sort_index())

Distribución de `nivel_estres` (solo valores dentro de la escala):

nivel_estres
0.0    109
1.0    172
2.0     69
3.0      4
Name: count, dtype: int64

### 4.5 Tabla de hallazgos y acciones propuestas

In [23]:
hallazgos = pd.DataFrame([
    ["perros", "Valores faltantes", f"edad_anios: {perros['edad_anios'].isna().sum()} nulos; primera_vez_guarderia: {perros['primera_vez_guarderia'].isna().sum()} nulos", "Reduce los datos de control disponibles", "Revisar la ficha original; imputar la edad con la mediana solo si es necesario"],
    ["perros", "Valores imposibles", f"{len(edad_invalida)} edades negativas o mayores a 25", "Distorsionan promedios", "Verificar en la ficha y corregir o marcar como nulo"],
    ["perros", "Duplicados", f"{int(perros.duplicated().sum())} fila(s) duplicada(s)", "Cuenta dos veces al mismo perro", "Eliminar con drop_duplicates()"],
    ["perros", "Categorías inconsistentes", f"sexo tiene {perros['sexo'].nunique()} valores distintos", "Agrupaciones incorrectas", "Normalizar con str.strip() y str.capitalize()"],
    ["perros", "Tipo de dato", "fecha_ingreso es de tipo texto", "No permite cálculos con fechas", "Convertir con pd.to_datetime(dayfirst=True)"],
    ["obs", "Valores faltantes", f"perros_en_espacio: {obs['perros_en_espacio'].isna().sum()} nulos; nivel_estres: {obs['nivel_estres'].isna().sum()} nulos; jadeo: {obs['jadeo'].isna().sum()} nulos", "Se pierden observaciones de las variables clave", "Completar desde la ficha de observación; excluir si no es posible"],
    ["obs", "Tipo de dato", f"perros_en_espacio es de tipo texto; {no_numericos.sum()} valores con texto", "Impide calcular correlaciones", "Convertir con pd.to_numeric(errors='coerce') y corregir manualmente"],
    ["obs", "Fuera de rango", f"{len(fuera_escala)} niveles de estrés fuera de 0-3; {len(dia_invalido)} día de estadía menor a 1", "Sesgan la relación estudiada", "Marcar como nulos y revisar la ficha"],
    ["obs", "Atípicos", f"{len(atipicos)} registro con más de 30 perros en el espacio", "Distorsiona la relación con el estrés", "Verificar si es error de digitación"],
    ["obs", "Integridad entre archivos", f"{len(sin_ficha)} observación sin ficha de ingreso", "No se puede unir con los datos del perro", "Revisar el identificador o excluir"],
    ["obs", "Duplicados", f"{int(obs.duplicated().sum())} fila(s) duplicada(s)", "Cuenta dos veces la misma observación", "Eliminar con drop_duplicates()"],
], columns=["Dataset", "Problema identificado", "Evidencia encontrada", "Posible impacto", "Acción propuesta"])
hallazgos

,Dataset,Problema identificado,Evidencia encontrada,Posible impacto,Acción propuesta
0,perros,Valores faltantes,edad_anios: 5 nulos; primera_vez_guarderia: 4 ...,Reduce los datos de control disponibles,Revisar la ficha original; imputar la edad con...
1,perros,Valores imposibles,2 edades negativas o mayores a 25,Distorsionan promedios,Verificar en la ficha y corregir o marcar como...
2,perros,Duplicados,1 fila(s) duplicada(s),Cuenta dos veces al mismo perro,Eliminar con drop_duplicates()
3,perros,Categorías inconsistentes,sexo tiene 4 valores distintos,Agrupaciones incorrectas,Normalizar con str.strip() y str.capitalize()
4,perros,Tipo de dato,fecha_ingreso es de tipo texto,No permite cálculos con fechas,Convertir con pd.to_datetime(dayfirst=True)
5,obs,Valores faltantes,perros_en_espacio: 12 nulos; nivel_estres: 6 n...,Se pierden observaciones de las variables clave,Completar desde la ficha de observación; exclu...
6,obs,Tipo de dato,perros_en_espacio es de tipo texto; 2 valores ...,Impide calcular correlaciones,Convertir con pd.to_numeric(errors='coerce') y...
7,obs,Fuera de rango,2 niveles de estrés fuera de 0-3; 1 día de est...,Sesgan la relación estudiada,Marcar como nulos y revisar la ficha
8,obs,Atípicos,1 registro con más de 30 perros en el espacio,Distorsiona la relación con el estrés,Verificar si es error de digitación
9,obs,Integridad entre archivos,1 observación sin ficha de ingreso,No se puede unir con los datos del perro,Revisar el identificador o excluir


---
## Conclusión

In [24]:
total_problemas = len(hallazgos)
md(f"Se identificaron **{total_problemas}** problemas de calidad entre los dos datasets. Ninguno impide continuar con el proyecto, "
   f"pero **antes de analizar la relación entre perros por espacio, días de estadía y estrés** hay que convertir `perros_en_espacio` a numérico, "
   f"tratar los valores fuera de la escala, eliminar duplicados y resolver los faltantes de las variables clave. "
   f"Después de esa preparación quedarían **{len(validas)}** observaciones utilizables como punto de partida.")

Se identificaron **11** problemas de calidad entre los dos datasets. Ninguno impide continuar con el proyecto, pero **antes de analizar la relación entre perros por espacio, días de estadía y estrés** hay que convertir `perros_en_espacio` a numérico, tratar los valores fuera de la escala, eliminar duplicados y resolver los faltantes de las variables clave. Después de esa preparación quedarían **338** observaciones utilizables como punto de partida.